# Mobilscan - 3D Gaussian Splatting Generátor 🚀
Ezzel a programmal az iPhone-oddal szkennelt adatokból valósághű 3D modellt készíthetsz!

**Lépések:**
1. Tömörítsd be az iPhone-odról lementett mappát egy **`Scan.zip`** fájlba (úgy, hogy a zipben egyből az `images` mappa és a `transforms.json` legyen benne).
2. Bal oldalon a **Mappa ikonra 📁** kattintva húzd be a `Scan.zip` fájlt a Colab fájljai közé.
3. Futtasd le sorban az alábbi kódblokkokat (a kis lejátszás gombra kattintva rajtuk)!

### 1. Rendszer előkészítése és NerfStudio telepítése (kb. 3-5 perc)
*A háttérben létrehozunk egy elszigetelt Python 3.10 környezetet, hogy kikerüljük a Colab legújabb frissítéseiből adódó hibákat.*

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!/usr/local/bin/uv python install 3.10
!/usr/local/bin/uv venv --python 3.10 /content/.venv
!/usr/local/bin/uv pip install --python /content/.venv "setuptools<70" wheel
!/usr/local/bin/uv pip install --python /content/.venv "numpy<2"
!/usr/local/bin/uv pip install --python /content/.venv torch==2.1.2 torchvision==0.16.2 --extra-index-url https://download.pytorch.org/whl/cu121
!/usr/local/bin/uv pip install --python /content/.venv ninja
!/usr/local/bin/uv pip install --python /content/.venv nerfstudio


### 2. Adatok kicsomagolása

In [ ]:
!rm -rf /content/ScanData
!mkdir /content/ScanData
!unzip -q /content/Scan.zip -d /content/ScanData/
print("✅ Adatok sikeresen kicsomagolva a /content/ScanData mappába!")

### 3. ARKit adatok konvertálása NerfStudio formátumra

In [ ]:
import json
import os
import shutil
from PIL import Image

def convert_arkit_to_nerfstudio(data_dir):
    input_json_path = os.path.join(data_dir, "transforms.json")
    
    if not os.path.exists(input_json_path):
        subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        if subdirs and os.path.exists(os.path.join(subdirs[0], "transforms.json")):
            data_dir = subdirs[0]
            input_json_path = os.path.join(data_dir, "transforms.json")
        else:
            print(f"❌ HIBA: Nem találom a transforms.json fájlt itt: {input_json_path}")
            return data_dir
            
    # Mentsük el az eredeti ARKit json-t, ha még nincs
    arkit_json_path = os.path.join(data_dir, "transforms_arkit.json")
    if os.path.exists(arkit_json_path):
        print("Már át van konvertálva, arkit mentés megvan.")
        input_json_path = arkit_json_path
    else:
        shutil.copy(input_json_path, arkit_json_path)
        
    with open(input_json_path, 'r') as f:
        data = json.load(f)
        
    frames = data.get("frames", [])
    if not frames:
        print("❌ HIBA: Nincsenek képkockák a transforms.json-ben!")
        return data_dir
        
    first_image_path = os.path.join(data_dir, frames[0]["file_path"])
    if not os.path.exists(first_image_path):
        print(f"❌ HIBA: Nem találom az első képet: {first_image_path}.")
        return data_dir
        
    with Image.open(first_image_path) as img:
        w, h = img.size
        
    intrinsics = frames[0]["intrinsics_matrix"]
    fl_x = intrinsics[0][0]
    fl_y = intrinsics[1][1]
    cx = intrinsics[2][0]
    cy = intrinsics[2][1]
    
    out_data = {
        "camera_model": "OPENCV",
        "orientation_override": "none",
        "fl_x": fl_x,
        "fl_y": fl_y,
        "cx": cx,
        "cy": cy,
        "w": w,
        "h": h,
        "frames": []
    }
    
    for frame in frames:
        matrix = frame["transform_matrix"]
        
        c2w = [
            [matrix[0][0], -matrix[1][0], -matrix[2][0], matrix[3][0]],
            [matrix[0][1], -matrix[1][1], -matrix[2][1], matrix[3][1]],
            [matrix[0][2], -matrix[1][2], -matrix[2][2], matrix[3][2]],
            [matrix[0][3], -matrix[1][3], -matrix[2][3], matrix[3][3]]
        ]
        
        out_frame = {
            "file_path": frame["file_path"],
            "transform_matrix": c2w,
            "fl_x": fl_x,
            "fl_y": fl_y,
            "cx": cx,
            "cy": cy
        }
        
        out_data["frames"].append(out_frame)
        
    output_json_path = os.path.join(data_dir, "transforms.json")
    with open(output_json_path, 'w') as f:
        json.dump(out_data, f, indent=4)
        
    print(f"✅ Sikeresen konvertáltam {len(frames)} képkockát NerfStudio formátumra!")
    return data_dir

actual_data_dir = convert_arkit_to_nerfstudio("/content/ScanData")
with open("/content/actual_data_dir.txt", "w") as f:
    f.write(actual_data_dir)

### 4. Tanítás (Gaussian Splatting generálás)
Elindítjuk a `splatfacto` AI modellt. Ez kb. 10-15 percet vesz igénybe a feltöltött képek számától függően.

In [ ]:
!DATA_DIR=$(cat /content/actual_data_dir.txt) && /content/.venv/bin/ns-train splatfacto --vis tensorboard --pipeline.datamanager.max-thread-workers 1 --pipeline.model.camera-optimizer.mode off --pipeline.model.cull-alpha-thresh 0.005 --pipeline.model.sh-degree 3 --timestamp "mobilscan_run" nerfstudio-data --data $DATA_DIR --downscale-factor 4


### 5. Modell exportálása és letöltése

In [ ]:
!CONFIG_FILE=$(find outputs -name "config.yml" | head -n 1) && /content/.venv/bin/ns-export gaussian-splat --load-config $CONFIG_FILE --output-dir /content/export
print("✅ Modell exportálva! Keresd a bal oldali Fájlok (Files) menüben a /content/export/splat.ply fájlt, kattints rá jobb gombbal és 'Download'!")
print("Ezután megnyithatod bármilyen Gaussian Splatting nézegetőben, pl: https://playcanvas.com/supersplat/editor")